In [ ]:
# Importing Libraries
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.io import read_image
import torchvision.transforms.functional as F_t
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets.utils import download_url
import matplotlib.pyplot as plt
from IPython import display
import cv2
from PIL import Image


try:
    from einops import rearrange
except ImportError:
    %pip install einops
    from einops import rearrange

In [ ]:
# creating Random Fourier Features
class RFF(nn.Module):
    def __init__(self, features_in, features_out, gamma): # gamma is the scaling factor
        super().__init__() # calling the __init__ method of the parent class
        self.B = torch.randn(features_in, features_out)*gamma # creating a random matrix of size in_features x out_features

    def forward(self, x): # defining the forward method
        return torch.hstack([torch.sin(torch.matmul(2*math.pi*x, self.B)), torch.cos(torch.matmul(2*math.pi*x, self.B))]) # returning the output of the RFF layer

In [ ]:
class RFF_SR(nn.Module): # defining the RFF_SR class
    def __init__(self, in_features, fourier_features, out_features, gamma): # defining the __init__ method
        super().__init__() # calling the __init__ method of the parent class
        self.net = nn.Sequential(RFF(in_features, fourier_features, gamma), nn.Linear(2*fourier_features, out_features)) 
        
    def forward(self, x): # defining the forward method
      return self.net(x) # returning the output of the RFF_SR layer

In [ ]:
def create_coord_map(image_path): # defining the create_coord_map function
        image = read_image(image_path) # reading the image
        h, w = image.shape[1], image.shape[2] # getting the height and width of the image
        image = image.float()/255 # normalizing the image
        image = image.permute(1, 2, 0) # changing the dimensions of the image
        # print(image)

        h_coords = torch.linspace(0, 1, steps=h) # creating a tensor of linearly spaced values from 0 to 1 with h steps
        w_coords = torch.linspace(0, 1, steps=w) # creating a tensor of linearly spaced values from 0 to 1 with w steps
        grid = torch.stack(torch.meshgrid(h_coords, w_coords), dim=-1) # stacking the meshgrid of h_coords and w_coords along the last dimension
        # print(h, w)

        return grid, image # returning the grid and the image

In [ ]:
image_path = "dog1.jpg" # path to the image
display.Image(image_path) # displaying the image



In [ ]:
print(cv2.imread(image_path).shape) # printing the shape of the image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # setting the device to cuda if it is available, else to cpu

grid, image = create_coord_map(image_path) # creating the grid and the image
# print(grid.shape, image.shape)
grid, image = grid.squeeze().to(device), image.squeeze().to(device) # removing the extra dimension and moving the grid and the image to the device


In [ ]:
percentage = 0.1 # percentage of missing points
boolean_array = torch.rand(100, 100) < percentage # create a boolean array of the same shape as the image

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 100) # create a tensor of linearly spaced values from -1 to 1 with 100 steps
y = torch.linspace(-1, 1, 100) # create a tensor of linearly spaced values from -1 to 1 with 100 steps
X, Y = torch.meshgrid(x, y) # create a meshgrid of x and y


# Apply the boolean mask to the image array
masked_image = image.clone() # create a copy of the image
masked_image[boolean_array, ] = -1 # set the values of the masked image to -1 where the boolean array is True

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {percentage*100}%")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape) # print the shapes of the training coordinates and the training rgb values
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device) # creating the SuperReso model
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01) # creating the optimizer
train_mse_arr = [] # creating an empty list to store the training mse values
train_psnr_arr = [] # creating an empty list to store the training psnr values

for iter in range(1, 101): # iterating 100 times
    SuperReso.train() # setting the model to training mode
    optimizer.zero_grad() # zeroing the gradients
    output = SuperReso(train_coords) # getting the output of the model
    train_loss = F.mse_loss(output, train_rgbs) # calculating the mse loss
    train_loss.backward() # backpropagating the loss
    optimizer.step() # updating the weights
    train_mse_arr.append(train_loss.item()) # appending the loss to the list
    train_psnr_arr.append(-10*torch.log10(train_loss)) # appending the psnr to the list

    with torch.no_grad(): # setting the requirement to calculate gradients to False
        prediction = SuperReso(test_coords) # getting the output of the model
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}") # printing the psnr and mse loss
    if iter % 10 == 0: # if the iteration is a multiple of 10
        fig, axes = plt.subplots(1, 2, figsize=(15, 5)) # creating a figure and axes
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu()) # displaying the prediction
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {percentage*100}%") # setting the title
        axes[1].imshow(image.cpu()) # displaying the original image
        axes[1].set_title(f"Original Image") # setting the title   
        plt.show() # displaying the plot

plt.figure() # creating a figure
plt.plot(train_mse_arr) # plotting the mse loss
plt.xlabel("Number of Iterations") # setting the x label
plt.ylabel("MSE Loss") # setting the y label
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {percentage*100}%") # setting the title
plt.show() # displaying the plot

for i in range(len(train_psnr_arr)): # iterating over the length of the list
    train_psnr_arr[i] = train_psnr_arr[i].item() # converting the tensor to a float
plt.figure() # creating a figure
plt.plot(train_psnr_arr) # plotting the psnr
plt.xlabel("Number of Iterations") # setting the x label
plt.ylabel("PSNR") # setting the y label
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {percentage*100}%") # setting the title
plt.show() # displaying the plot


Using the same parameterized code with a different value of percentage

In [ ]:
percentage = 0.2
boolean_array = torch.rand(100, 100) < percentage

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 100)
y = torch.linspace(-1, 1, 100)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {percentage*100}%")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 101):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {percentage*100}%")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()


In [ ]:
percentage = 0.3
boolean_array = torch.rand(100, 100) < percentage

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 100)
y = torch.linspace(-1, 1, 100)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {percentage*100}%")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 101):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {percentage*100}%")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()


In [ ]:
percentage = 0.4
boolean_array = torch.rand(100, 100) < percentage

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 100)
y = torch.linspace(-1, 1, 100)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {percentage*100}%")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 101):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {percentage*100}%")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()


In [ ]:
percentage = 0.5
boolean_array = torch.rand(100, 100) < percentage

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 100)
y = torch.linspace(-1, 1, 100)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {percentage*100}%")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 101):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {percentage*100}%")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()


In [ ]:
percentage = 0.6
boolean_array = torch.rand(100, 100) < percentage

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 100)
y = torch.linspace(-1, 1, 100)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {percentage*100}%")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 101):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {percentage*100}%")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()


In [ ]:
percentage = 0.7
boolean_array = torch.rand(100, 100) < percentage

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 100)
y = torch.linspace(-1, 1, 100)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {percentage*100}%")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 101):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {percentage*100}%")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()


In [ ]:
percentage = 0.8
boolean_array = torch.rand(100, 100) < percentage

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 100)
y = torch.linspace(-1, 1, 100)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {percentage*100}%")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 101):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {percentage*100}%")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()


In [ ]:
percentage = 0.9
boolean_array = torch.rand(100, 100) < percentage

# Create a 3D grid array of coordinates
x = torch.linspace(-1, 1, 100)
y = torch.linspace(-1, 1, 100)
X, Y = torch.meshgrid(x, y)


# Apply the boolean mask to the image array
masked_image = image.clone()
masked_image[boolean_array, ] = -1

# Display the masked image without missing points
plt.figure()
plt.imshow(masked_image)
plt.title(f"Image for degradation by {percentage*100}%")
plt.axis('off')
plt.show()


train_coords, train_rgbs = grid[masked_image[:,:,0]!=-1].reshape(-1, 2), image[masked_image!=-1].reshape(-1, 3)  # use every other pixel for training
print(train_coords.shape, train_rgbs.shape)
test_coords = grid.reshape(-1, 2) # use all the pixels for evaluation
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device)
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01)
train_mse_arr = []
train_psnr_arr = []

for iter in range(1, 101):
    SuperReso.train()
    optimizer.zero_grad()
    output = SuperReso(train_coords)
    train_loss = F.mse_loss(output, train_rgbs)
    train_loss.backward()
    optimizer.step()
    train_mse_arr.append(train_loss.item())
    train_psnr_arr.append(-10*torch.log10(train_loss))

    with torch.no_grad():
        prediction = SuperReso(test_coords)
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}")
    if iter % 10 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].imshow(prediction.reshape(image.shape[0], image.shape[1], image.shape[2]).cpu())
        axes[0].set_title(f"Prediction Result in Iteration {iter} for degradation by {percentage*100}%")
        axes[1].imshow(image.cpu())
        axes[1].set_title(f"Original Image")
        plt.show()

plt.figure()
plt.plot(train_mse_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("MSE Loss")
plt.title(f"MSE Loss vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()

for i in range(len(train_psnr_arr)):
    train_psnr_arr[i] = train_psnr_arr[i].item()
plt.figure()
plt.plot(train_psnr_arr)
plt.xlabel("Number of Iterations")
plt.ylabel("PSNR")
plt.title(f"PSNR vs Number of Iterations during Training for degradation by {percentage*100}%")
plt.show()
